Testing OpenCV for frame extraction

In [5]:
import cv2

video_path = r"C:\Users\jonas\OneDrive\Desktop\Speciale\Dataset\nutrition5k_dataset\nutrition5k_dataset\imagery\side_angles\dish_1550704750\camera_A.h264"

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("OpenCV cannot open this video.")
else:
    print("Video opened successfully.")

    ret, frame = cap.read()
    if ret:
        print("Frame shape:", frame.shape)
    else:
        print("Could not read frame.")

cap.release()

Video opened successfully.
Frame shape: (1080, 1920, 3)


Dataset creation

When we just extract the frames naturally then they come out flipped so in order to preserve consistency this script just flips them 180 degrees instead.
The frame extracting has also been limited to just the first and middle frame, it was not the end frame due to the camera starting and ending at the same location

In [11]:
import shutil
from pathlib import Path
import cv2


# =========================================================
# Parameter setting
# =========================================================

DATASET_ROOT = Path(r"C:\Users\jonas\OneDrive\Desktop\Speciale\Dataset\nutrition5k_dataset\nutrition5k_dataset")
OUTPUT_ROOT  = Path(r"C:\Users\jonas\OneDrive\Desktop\Speciale\Dataset\nutrition5k_compactV3")

INCLUDE_SIDE_ANGLES = True

# Save exactly 2 frames per camera: start + middle
FRAMES_PER_CAMERA = 2

# Default rotation behavior for side-angle frames
ROTATE_SIDE_180_DEFAULT = True

# After (and including) this dish timestamp, stop rotating side-angle frames
STOP_ROTATING_AFTER_DISH_TS = 1562602589
#It seemed that they themselves identified the error so started flipping the images themselves
#so after this dish id we will have to stop flipping the images in order to ensure consistency.


# Testing the script
LIMIT_DISHES = None

# =========================================================
def safe_mkdir(path: Path):
    path.mkdir(parents=True, exist_ok=True)


def find_file_any_ext(folder: Path, basename: str):
    """
    Finds files like:
        rgb
        rgb.png
        rgb.jpg
    """
    exact = folder / basename
    if exact.exists():
        return exact

    matches = list(folder.glob(basename + ".*"))
    if matches:
        matches = sorted(matches, key=lambda x: (x.suffix.lower() != ".png", x.name))
        return matches[0]

    return None


def copy_overhead_files(overhead_dir: Path, out_dir: Path):
    """
    Copies:
      - rgb
      - depth_color
      - depth_raw
    into:
      out_dir/overhead/
    """
    out_overhead = out_dir / "overhead"
    safe_mkdir(out_overhead)

    for name in ["rgb", "depth_color", "depth_raw"]:
        src = find_file_any_ext(overhead_dir, name)
        if src is None:
            print(f"[WARNING] Missing {name} in {overhead_dir.name}")
            continue

        dst = out_overhead / src.name
        if not dst.exists():
            shutil.copy2(src, dst)


def rotate_if_needed(frame, rotate_180: bool):
    if frame is None:
        return None
    if rotate_180:
        return cv2.rotate(frame, cv2.ROTATE_180)
    return frame


def should_rotate_side_angles(dish_id: str) -> bool:
    """
    Option B: rotate only up to a cutoff dish timestamp.
    For dish_1562602589:
      - if STOP_ROTATING_AFTER_DISH_TS == 1562602589
      - then dishes with ts >= 1562602589 will NOT be rotated
    """
    if not ROTATE_SIDE_180_DEFAULT:
        return False

    try:
        dish_ts = int(dish_id.replace("dish_", ""))
    except ValueError:
        # If parsing fails, fall back to default behavior
        return ROTATE_SIDE_180_DEFAULT

    return dish_ts < STOP_ROTATING_AFTER_DISH_TS


def extract_start_and_middle_frames(video_path: Path, out_dir: Path, rotate_180: bool = True):
    """
    Extracts exactly 2 frames from a video:
      - first frame (start)
      - middle frame (by frame index)
    Works reliably even when CAP_PROP_FRAME_COUNT is not trustworthy by doing a first pass to count frames.
    """
    safe_mkdir(out_dir)

    # -------- Pass 1: count frames --------
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print(f"[WARNING] Could not open {video_path.name}")
        return

    total = 0
    while True:
        ret, _ = cap.read()
        if not ret:
            break
        total += 1
    cap.release()

    if total == 0:
        print(f"[WARNING] No frames read from {video_path.name}")
        return

    mid_idx = total // 2  # middle (for even counts, this is the "upper-middle")

    # -------- Pass 2: read again and save start + middle --------
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print(f"[WARNING] Could not reopen {video_path.name}")
        return

    idx = 0
    saved_start = False
    saved_mid = False

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if idx == 0 and not saved_start:
            frame_s = rotate_if_needed(frame, rotate_180)
            out_path = out_dir / "frame_start.jpg"
            cv2.imwrite(str(out_path), frame_s)
            saved_start = True

        if idx == mid_idx and not saved_mid:
            frame_m = rotate_if_needed(frame, rotate_180)
            out_path = out_dir / "frame_mid.jpg"
            cv2.imwrite(str(out_path), frame_m)
            saved_mid = True

        if saved_start and saved_mid:
            break

        idx += 1

    cap.release()

    if not (saved_start and saved_mid):
        print(f"[WARNING] Only saved start={saved_start}, mid={saved_mid} for {video_path.name} (total frames={total})")


def process_side_angles(side_root: Path, dish_id: str, out_dish_dir: Path):
    """
    For a dish:
      side_angles/dish_xxx/camera_A.h264
      side_angles/dish_xxx/camera_B.h264
      ...
    Extract start+middle frames into:
      out_dish_dir/side_angles/camera_A/frame_start.jpg
      out_dish_dir/side_angles/camera_A/frame_mid.jpg
    """
    dish_side_dir = side_root / dish_id
    if not dish_side_dir.exists():
        return

    camera_files = sorted(dish_side_dir.glob("camera_*.h264"))
    if not camera_files:
        camera_files = sorted(dish_side_dir.glob("*.h264"))

    if not camera_files:
        return

    out_side = out_dish_dir / "side_angles"
    safe_mkdir(out_side)

    rotate_180 = should_rotate_side_angles(dish_id)

    for video in camera_files:
        cam_name = video.stem  # camera_A, camera_B, ...
        out_cam = out_side / cam_name
        extract_start_and_middle_frames(video, out_cam, rotate_180=rotate_180)


# =========================================================
# Dataset creation
# =========================================================

overhead_root = DATASET_ROOT / "imagery" / "realsense_overhead"
side_root     = DATASET_ROOT / "imagery" / "side_angles"

safe_mkdir(OUTPUT_ROOT)

dish_dirs = sorted([
    p for p in overhead_root.iterdir()
    if p.is_dir() and p.name.startswith("dish_")
])

print(f"Found {len(dish_dirs)} dishes.")

# Limit dishes for testing if desired
if LIMIT_DISHES is not None:
    dish_dirs = dish_dirs[:LIMIT_DISHES]

for dish_dir in dish_dirs:
    dish_id = dish_dir.name
    out_dish_dir = OUTPUT_ROOT / dish_id
    safe_mkdir(out_dish_dir)

    # 1) Copy overhead images (rgb + depth_color + depth_raw)
    copy_overhead_files(dish_dir, out_dish_dir)

    # 2) Extract side angle frames (optional)
    if INCLUDE_SIDE_ANGLES:
        process_side_angles(side_root, dish_id, out_dish_dir)

print("Dataset creation complete.")

Found 3493 dishes.
[WARNING] Missing rgb in dish_1557862384
[WARNING] Missing depth_color in dish_1557862384
[WARNING] Missing depth_raw in dish_1557862384
[WARNING] Missing rgb in dish_1558109511
[WARNING] Missing depth_color in dish_1558109511
[WARNING] Missing depth_raw in dish_1558109511
[WARNING] Missing rgb in dish_1558109714
[WARNING] Missing depth_color in dish_1558109714
[WARNING] Missing depth_raw in dish_1558109714
[WARNING] No frames read from camera_D.h264
Dataset creation complete.


Testing all dishes has been added

Original dataset

In [12]:
overhead_root = Path(
    r"C:\Users\jonas\OneDrive\Desktop\Speciale\Dataset\nutrition5k_dataset\nutrition5k_dataset\imagery\realsense_overhead"
)

dish_count = len([
    p for p in overhead_root.iterdir()
    if p.is_dir() and p.name.startswith("dish_")
])

print(f"Number of dishes in original dataset: {dish_count}")

Number of dishes in original dataset: 3493


Completed dataset

In [13]:
from pathlib import Path

folder = Path(r"C:\Users\jonas\OneDrive\Desktop\Speciale\Dataset\nutrition5k_compactV3")

dish_count = len([
    p for p in folder.iterdir()
    if p.is_dir() and p.name.startswith("dish_")
])

print(f"Number of dishes: {dish_count}")

Number of dishes: 3493
